# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset ([Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/)) using the `mlcroissant` library according to the Croissant schema.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json), providing machine-readable metadata and access instructions.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata so we can inspect its structure and contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)

# Display basic metadata from the Dataset object
print(f"Dataset name: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}\n\nVersion: {getattr(dataset.metadata, 'version', 'N/A')}")

## 2. Data Overview
Explore the dataset: examine available record sets, fields, and their Croissant `@id`s.

In [ ]:
# List all record sets in the dataset using their @id
print("Available Record Sets:\n---------------------")
for rs in dataset.metadata.record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction

Extract tabular data from a specific record set as a DataFrame for analysis. 

We identify a principal record set and extract all rows/columns for further processing.

**Note:** All references use Croissant `@id` values for record sets and fields.

In [ ]:
# Choose the main tabular record set for analysis. List all record set @ids.
record_sets = [rs.id for rs in dataset.metadata.record_sets]
print(f"Detected record sets by @id: {record_sets}")

# For this dataset, we use the main record set, which typically lists proteins and their attributes.
# We'll pick the first record set for demonstration; feel free to change this index for others.
record_set_id = record_sets[0]

# Load all records from the record set as a DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

print(f"Columns in DataFrame for record set {record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply some basic processing and transformations to numeric columns, following proper field selection by Croissant `@id`.

We'll filter records, normalize a numeric column, and group by a categorical variable.

In [ ]:
# List all field ids and select a numeric field and a suitable group/category field for EDA.
fields = {field.id: field for field in dataset.metadata.record_sets[0].fields}

# Identify numeric fields
numeric_fields = [fid for fid, field in fields.items() if field.data_type in ("Number", "Float", "Integer") and fid in df.columns]
print(f"Numeric fields: {numeric_fields}")

# Select a numeric field - e.g., molecular weight (MW) or peptide count (pick one that exists)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    print("No numeric fields found. Please adjust your selection.")
    numeric_field_id = df.select_dtypes('number').columns[0]

# Try to find a group-by categorical field (commonly accession or protein description fields, or sample id)
categorical_fields = [fid for fid, field in fields.items() if field.data_type == "Text" and fid in df.columns]
if categorical_fields:
    group_field_id = categorical_fields[0]
else:
    group_field_id = df.select_dtypes('object').columns[0]

print(f"Using numeric field: {numeric_field_id}")
print(f"Grouping by: {group_field_id}")

# Set a threshold for filtering
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the chosen field and compute mean values
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped data by {group_field_id}, aggregating mean {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Plot distributions for quantitative fields and relationships between attributes (using the Croissant `@id` selection).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histograms for the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If another numeric field is available, plot a scatterplot
if len(numeric_fields) > 1:
    plt.figure(figsize=(6, 6))
    sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
    plt.xlabel(numeric_fields[0])
    plt.ylabel(numeric_fields[1])
    plt.title(f"Scatter plot of {numeric_fields[0]} vs. {numeric_fields[1]}")
    plt.show()


## 6. Conclusion

In this notebook we loaded the FAIR² dataset Croissant package using `mlcroissant`, obtained an overview of its available record sets and fields (referenced strictly by Croissant `@id`), extracted principal tabular data as DataFrames for analysis, performed basic filtering and normalization of numeric attributes, and visualized the distributions and relationships in the data.

This workflow demonstrates how to use Croissant metadata in reproducible ML/DS pipelines. Refer to the [FAIR² data record](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) for domain interpretation and citation.